# Santorini V3 Kaggle Training

V3 learns worker placement and normal play from an empty board. Active checkpoints and replay data live in `/kaggle/tmp`, while compact resume artifacts and telemetry are exported to `/kaggle/working`. Kaggle commits preserve only `/kaggle/working`, whose saved-output allowance is 5 GB.

## 0. P100 PyTorch compatibility guard

In [ ]:
import subprocess, sys
try:
    gpu_name = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip()
except Exception as exc:
    gpu_name = ""
    print("Could not query GPU:", exc)
print("GPU:", gpu_name or "none")
if "P100" in gpu_name:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "--force-reinstall", "--no-cache-dir", "torch==2.10.0", "--index-url", "https://download.pytorch.org/whl/cu126"])
    print("P100 wheel installed. Restart the kernel if torch was already imported.")

## 1. Repository and runtime

In [ ]:
from kaggle_secrets import UserSecretsClient
import os, torch

user_secrets = UserSecretsClient()
github_token = user_secrets.get_secret("GITHUB_PAT")

REPO_URL = "https://github.com/Luminous9/alpha-zero-custom.git"
REPO_DIR = "/kaggle/working/alpha-zero-general"
print("PyTorch:", torch.__version__, "CUDA:", torch.cuda.is_available())
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
%cd /kaggle/working

import base64
import os
import subprocess

auth = base64.b64encode(
    f"x-access-token:{github_token}".encode()
).decode()

git_env = os.environ.copy()
git_env.update({
    "GIT_CONFIG_COUNT": "1",
    "GIT_CONFIG_KEY_0": "http.extraHeader",
    "GIT_CONFIG_VALUE_0": f"Authorization: Basic {auth}",
    "GIT_TERMINAL_PROMPT": "0",
})

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    # Important if this directory was previously cloned from the old fork.
    subprocess.run(
        ["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL],
        check=True,
    )
    subprocess.run(
        ["git", "-C", REPO_DIR, "pull", "--ff-only"],
        check=True,
        env=git_env,
    )
else:
    subprocess.run(
        ["git", "clone", REPO_URL, REPO_DIR],
        check=True,
        env=git_env,
    )

%cd $REPO_DIR
!pip install --quiet coloredlogs tqdm tensorboard

## 2. Scratch, durable export, and optional resume input

In [ ]:
import glob, os, shutil
SCRATCH_ROOT = "/kaggle/tmp/santorini-v3"
CHECKPOINT = os.path.join(SCRATCH_ROOT, "checkpoints")
DURABLE_ROOT = "/kaggle/working/Santorini-AZ"
EXPORT_DIR = os.path.join(DURABLE_ROOT, "export")
TELEMETRY_DIR = os.path.join(DURABLE_ROOT, "telemetry")
for path in (CHECKPOINT, EXPORT_DIR, TELEMETRY_DIR): os.makedirs(path, exist_ok=True)
os.environ.update(CHECKPOINT=CHECKPOINT, EXPORT_DIR=EXPORT_DIR, TELEMETRY_DIR=TELEMETRY_DIR)
RESUME_CHECKPOINT_SOURCE = ""  # e.g. /kaggle/input/<dataset>/export
if RESUME_CHECKPOINT_SOURCE:
    if not os.path.isdir(RESUME_CHECKPOINT_SOURCE): raise FileNotFoundError(RESUME_CHECKPOINT_SOURCE)
    for source in glob.glob(os.path.join(RESUME_CHECKPOINT_SOURCE, "*")):
        destination = os.path.join(CHECKPOINT, os.path.basename(source))
        shutil.copytree(source, destination, dirs_exist_ok=True) if os.path.isdir(source) else shutil.copy2(source, destination)
    print("Resume artifacts copied to scratch.")
print("Scratch:", CHECKPOINT)
print("Durable export:", EXPORT_DIR)
print("Telemetry:", TELEMETRY_DIR)

## 3. Live TensorBoard
Run this before training. The iframe reads event files that V3 flushes after every iteration.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /kaggle/working/Santorini-AZ/telemetry --reload_interval 15

## 4. Training configuration
Attach the optional 500-position reference suite and fixed V1 anchor as Kaggle Datasets. Set `REFERENCE_SUITE` and `ANCHOR_CHECKPOINT` to their dataset directories or exact files; leave either empty to disable it.

In [ ]:
from datetime import datetime, timezone
RUN_STARTED_AT = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
RUN_SEED = int(RUN_STARTED_AT) % (2**32 - 1)
REFERENCE_SUITE = ""  # May be the attached dataset directory or the exact .npz file.
ANCHOR_CHECKPOINT = ""  # May be the attached V1 dataset directory or exact best.pth.tar file.
if REFERENCE_SUITE:
    from santorini.SantoriniTelemetry import resolve_reference_suite_path
    REFERENCE_SUITE = resolve_reference_suite_path(REFERENCE_SUITE)
    print("Reference suite:", REFERENCE_SUITE)
if ANCHOR_CHECKPOINT:
    from main_santorini import resolve_anchor_checkpoint_path
    ANCHOR_CHECKPOINT = resolve_anchor_checkpoint_path(ANCHOR_CHECKPOINT)
    print("Fixed V1 anchor:", ANCHOR_CHECKPOINT)
CHUNK_ITERATIONS = 10  # Set to 40 or 90 for a longer uninterrupted training chunk.
NUM_SELF_PLAY_GAMES = 240  # Doubles Run 6's outcomes for about 10% more nominal simulations.
NUM_MCTS_SIMS = 96
SEARCH_MODE = "gumbel"  # Current V3 continuation mode; use "puct" for a control branch.
GUMBEL_MAX_CONSIDERED_ACTIONS = 16
GUMBEL_SCALE = 1.0
GUMBEL_PLACEMENT_SCALE = 1.5  # Base placement scale, used by 90% of self-play games.
PLACEMENT_SCALE_EXPLORATION_PROBABILITY = 0.10
PLACEMENT_EXPLORATION_GUMBEL_SCALE = 2.25  # Used for the other 10% of self-play games.
EVALUATION_GUMBEL_SCALE = 0.0  # Deterministic standard-play milestones and anchors.
EVALUATION_GUMBEL_PLACEMENT_SCALE = 1.0  # Reproducibly varied placement milestones.
ROOT_SYMMETRY_SAMPLES = 2  # Standard self-play roots; interior leaves still use one random orientation.
PLACEMENT_ROOT_SYMMETRY_SAMPLES = 8  # All D4 orientations for the four placement roots.
EVALUATION_ROOT_SYMMETRY_SAMPLES = 8  # Stable standard-play milestone/anchor roots.
EVALUATION_PLACEMENT_ROOT_SYMMETRY_SAMPLES = 8
SYMMETRY_CONSISTENCY_FRACTION = 0.25  # Pair this fraction of each batch with another D4 orientation.
SYMMETRY_CONSISTENCY_POLICY_WEIGHT = 0.10  # Next symmetry experiment; mapped-back policy Jensen-Shannon loss.
SYMMETRY_CONSISTENCY_VALUE_WEIGHT = 0.10  # Next symmetry experiment; value agreement mean-squared error.
SYMMETRY_TELEMETRY_SAMPLE_SIZE = 64  # Fixed held-out suite, stratified across four game phases.
INFERENCE_CACHE_SIZE = 4096  # Exact-board predictions retained only for the current move.
PLAYOUT_CAP_FULL_PROBABILITY = 0.25
PLAYOUT_CAP_FAST_SIMS = 32
POLICY_TARGET_TEMPERATURE = 1.0  # Keep replay targets soft even after action selection becomes greedy.
REPLAY_REUSE = 16.0  # Training draws requested per newly generated, non-validation position.
VALIDATION_FRACTION = 0.05  # Stable board-hash split; reconstructed exactly after resume.
LEARNING_RATE = 0.0003
WEIGHT_DECAY = 0.0001
LR_SCHEDULE = "200:0.0001,400:0.00003"  # Absolute iteration numbers; use "none" to disable.
RESUME_ITERATION_OVERRIDE = None  # Recovery only; e.g. 50 if checkpoint metadata is known to be wrong.
if CHUNK_ITERATIONS < 1: raise ValueError("CHUNK_ITERATIONS must be at least 1")
COMMON_ARGS = [
    sys.executable, "main_santorini.py", "--architecture", "v3", "--training-mode", "latest",
    "--preset", "local", "--checkpoint", CHECKPOINT, "--telemetry-dir", TELEMETRY_DIR,
    "--compact-replay", "--checkpoint-examples-to-keep", "0",
    "--self-play-batch-size", "128", "--batch-size", "512",
    "--num-iters", str(CHUNK_ITERATIONS), "--num-eps", str(NUM_SELF_PLAY_GAMES), "--num-mcts-sims", str(NUM_MCTS_SIMS),
    "--search-mode", SEARCH_MODE, "--gumbel-max-considered-actions", str(GUMBEL_MAX_CONSIDERED_ACTIONS),
    "--gumbel-scale", str(GUMBEL_SCALE),
    "--gumbel-placement-scale", str(GUMBEL_PLACEMENT_SCALE),
    "--placement-scale-exploration-probability", str(PLACEMENT_SCALE_EXPLORATION_PROBABILITY),
    "--placement-exploration-gumbel-scale", str(PLACEMENT_EXPLORATION_GUMBEL_SCALE),
    "--evaluation-gumbel-scale", str(EVALUATION_GUMBEL_SCALE),
    "--evaluation-gumbel-placement-scale", str(EVALUATION_GUMBEL_PLACEMENT_SCALE),
    "--playout-cap-randomization", "--playout-cap-full-probability", str(PLAYOUT_CAP_FULL_PROBABILITY),
    "--playout-cap-fast-sims", str(PLAYOUT_CAP_FAST_SIMS),
    "--epochs", "3", "--max-train-steps", "1500", "--replay-reuse", str(REPLAY_REUSE),
    "--validation-fraction", str(VALIDATION_FRACTION), "--history-iters", "20",
    "--optimizer", "adamw", "--learning-rate", str(LEARNING_RATE),
    "--weight-decay", str(WEIGHT_DECAY), "--lr-schedule", LR_SCHEDULE,
    "--symmetry-augmentation", "on-the-fly",
    "--symmetry-consistency-fraction", str(SYMMETRY_CONSISTENCY_FRACTION),
    "--symmetry-consistency-policy-weight", str(SYMMETRY_CONSISTENCY_POLICY_WEIGHT),
    "--symmetry-consistency-value-weight", str(SYMMETRY_CONSISTENCY_VALUE_WEIGHT),
    "--symmetry-telemetry-sample-size", str(SYMMETRY_TELEMETRY_SAMPLE_SIZE),
    "--inference-cache-size", str(INFERENCE_CACHE_SIZE),
    "--root-symmetry-samples", str(ROOT_SYMMETRY_SAMPLES),
    "--placement-root-symmetry-samples", str(PLACEMENT_ROOT_SYMMETRY_SAMPLES),
    "--evaluation-root-symmetry-samples", str(EVALUATION_ROOT_SYMMETRY_SAMPLES),
    "--evaluation-placement-root-symmetry-samples", str(EVALUATION_PLACEMENT_ROOT_SYMMETRY_SAMPLES),
    "--milestone-interval", "20",
    "--telemetry-match-games", "40", "--telemetry-placement-games", "40",
    "--telemetry-placement-temperature", "1.0", "--telemetry-opening-seed", "20260715",
    "--placement-temperature", "1.0", "--policy-target-temperature", str(POLICY_TARGET_TEMPERATURE),
    "--dirichlet-alpha", "0.30", "--dirichlet-epsilon", "0.25",
    "--seed", str(RUN_SEED), "--quiet",
]
if REFERENCE_SUITE: COMMON_ARGS += ["--reference-suite", REFERENCE_SUITE]
if ANCHOR_CHECKPOINT: COMMON_ARGS += ["--anchor-checkpoint", ANCHOR_CHECKPOINT, "--anchor-architecture", "v1", "--anchor-interval", "10", "--anchor-games", "40", "--anchor-mcts-sims", "64"]
print("Seed:", RUN_SEED)
print("Iterations in this chunk:", CHUNK_ITERATIONS)
print("Self-play MCTS simulations:", NUM_MCTS_SIMS)
print("Search mode:", SEARCH_MODE)
print("Gumbel standard / placement scales:", GUMBEL_SCALE, GUMBEL_PLACEMENT_SCALE)
print("Exploratory placement mix (probability / scale):", PLACEMENT_SCALE_EXPLORATION_PROBABILITY, PLACEMENT_EXPLORATION_GUMBEL_SCALE)
print("Evaluation Gumbel standard / placement scales:", EVALUATION_GUMBEL_SCALE, EVALUATION_GUMBEL_PLACEMENT_SCALE)
print("Root symmetry samples (self-play std/place; eval std/place):", ROOT_SYMMETRY_SAMPLES, PLACEMENT_ROOT_SYMMETRY_SAMPLES, EVALUATION_ROOT_SYMMETRY_SAMPLES, EVALUATION_PLACEMENT_ROOT_SYMMETRY_SAMPLES)
print("Symmetry consistency (fraction / policy / value):", SYMMETRY_CONSISTENCY_FRACTION, SYMMETRY_CONSISTENCY_POLICY_WEIGHT, SYMMETRY_CONSISTENCY_VALUE_WEIGHT)
print("Fixed symmetry telemetry positions:", SYMMETRY_TELEMETRY_SAMPLE_SIZE)
print("Per-move exact inference cache entries:", INFERENCE_CACHE_SIZE)
print("Self-play games:", NUM_SELF_PLAY_GAMES)
print("Playout-cap full probability / fast sims:", PLAYOUT_CAP_FULL_PROBABILITY, PLAYOUT_CAP_FAST_SIMS)
print("Replay policy-target temperature:", POLICY_TARGET_TEMPERATURE)
print("Replay reuse / validation fraction:", REPLAY_REUSE, VALIDATION_FRACTION)
print("Optimizer / learning rate / schedule:", "AdamW", LEARNING_RATE, LR_SCHEDULE)

## 5A. Fresh training chunk
Runs only when `RESUME_CHECKPOINT_SOURCE` is empty. It is safe to use Run All; this cell skips itself for a resume run.

In [ ]:
if RESUME_CHECKPOINT_SOURCE:
    print("Skipping 5A: resume source is configured.")
else:
    subprocess.run(COMMON_ARGS, check=True)

## 5B. Resume a training chunk
Runs only when `RESUME_CHECKPOINT_SOURCE` is configured. It restores weights, optimizer state, RNG state, iteration metadata, and compact replay history.

In [ ]:
if not RESUME_CHECKPOINT_SOURCE:
    print("Skipping 5B: no resume source is configured.")
else:
    required = ["latest-training.pth.tar", "latest.examples.npz"]
    missing = [name for name in required if not os.path.isfile(os.path.join(CHECKPOINT, name))]
    if missing: raise FileNotFoundError("Resume source is missing: " + ", ".join(missing))
    checkpoint_path = os.path.join(CHECKPOINT, "latest-training.pth.tar")
    try:
        resume_payload = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    except TypeError:
        resume_payload = torch.load(checkpoint_path, map_location="cpu")
    stored_iteration = (resume_payload.get("training_metadata") or {}).get("iteration")
    del resume_payload
    effective_iteration = RESUME_ITERATION_OVERRIDE if RESUME_ITERATION_OVERRIDE is not None else stored_iteration
    print("Checkpoint iteration metadata:", stored_iteration)
    print("Effective resume iteration:", effective_iteration)
    if effective_iteration is None: raise ValueError("Checkpoint has no iteration metadata; set RESUME_ITERATION_OVERRIDE.")
    resume_args = COMMON_ARGS + ["--load-folder", CHECKPOINT, "--load-file", "latest-training.pth.tar", "--load-model", "--load-examples"]
    if RESUME_ITERATION_OVERRIDE is not None: resume_args += ["--start-iteration", str(RESUME_ITERATION_OVERRIDE)]
    print("Launching resume after iteration", effective_iteration)
    subprocess.run(resume_args, check=True)

## 6. Export a compact resumable snapshot
Run after every completed chunk. Scratch files are not preserved by a Kaggle commit.

In [ ]:
def directory_bytes(path):
    return sum(os.path.getsize(os.path.join(root, name)) for root, _, names in os.walk(path) for name in names)

for name in os.listdir(EXPORT_DIR):
    path = os.path.join(EXPORT_DIR, name)
    shutil.rmtree(path) if os.path.isdir(path) else os.remove(path)
for pattern in ("latest-training.pth.tar", "latest.pth.tar", "latest.examples.npz", "checkpoint_*.pth.tar", "symmetry_telemetry_suite.npz"):
    for source in glob.glob(os.path.join(CHECKPOINT, pattern)):
        shutil.copy2(source, os.path.join(EXPORT_DIR, os.path.basename(source)))
size_gb = directory_bytes(DURABLE_ROOT) / 1024**3
print(f"Durable output size: {size_gb:.3f} GB")
if size_gb > 4.5: raise RuntimeError("Durable export exceeds the 4.5 GB safety limit.")
if size_gb > 4.0: print("WARNING: durable output is approaching Kaggle's 5 GB limit.")
print(sorted(os.listdir(EXPORT_DIR)))

## 7. Fallback telemetry plots
Use this if the live TensorBoard iframe is unavailable or after a chunk completes.

In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt
telemetry_path = os.path.join(TELEMETRY_DIR, "telemetry.jsonl")
records = [json.loads(line) for line in open(telemetry_path) if line.strip()]
metrics = pd.DataFrame(records)
display(metrics.tail(10))
groups = [
    ["iteration_policy_loss", "iteration_value_loss", "iteration_total_loss"],
    ["symmetry_consistency_policy_js", "symmetry_consistency_value_mse", "symmetry_consistency_weighted_loss"],
    ["placement_symmetry_consistency_policy_js", "standard_symmetry_consistency_policy_js"],
    ["placement_symmetry_consistency_value_mse", "standard_symmetry_consistency_value_mse"],
    ["symmetry_placement_value_mean_orbit_range", "symmetry_all_standard_value_mean_orbit_range", "symmetry_early_value_mean_orbit_range", "symmetry_middle_value_mean_orbit_range", "symmetry_late_value_mean_orbit_range"],
    ["symmetry_placement_value_sign_disagreement_rate", "symmetry_all_standard_value_sign_disagreement_rate"],
    ["symmetry_placement_value_orientation_mse", "symmetry_placement_value_ensemble_mse", "symmetry_placement_value_symmetry_excess_mse", "symmetry_all_standard_value_orientation_mse", "symmetry_all_standard_value_ensemble_mse", "symmetry_all_standard_value_symmetry_excess_mse"],
    ["symmetry_placement_policy_mean_orbit_total_variation", "symmetry_all_standard_policy_mean_orbit_total_variation"],
    ["inference_requested_evaluations", "inference_executed_evaluations", "inference_reused_evaluations", "inference_reuse_rate"],
    ["placement_validation_policy_kl", "standard_validation_policy_kl"],
    ["placement_validation_value_loss", "standard_validation_value_loss"],
    ["placement_validation_policy_top1_accuracy", "standard_validation_policy_top1_accuracy", "placement_validation_value_sign_accuracy", "standard_validation_value_sign_accuracy"],
    ["average_standard_plies", "median_standard_plies", "first_player_win_rate"],
    ["reference_policy_kl", "reference_value_mse", "reference_top1_accuracy"],
    ["placement_base_scale_games", "placement_exploratory_scale_games", "placement_exploratory_scale_game_rate"],
    ["unique_completed_openings", "symmetry_unique_completed_openings", "most_frequent_completed_opening_rate", "most_frequent_completed_opening_symmetry_rate"],
    ["placement_1_selection_entropy", "placement_3_selection_entropy"],
    ["p1_placement_mean_center_distance", "p1_winner_mean_center_distance", "p1_loser_mean_center_distance"],
    ["p1_placement_mean_worker_separation", "p1_winner_mean_worker_separation", "p1_loser_mean_worker_separation"],
    ["p1_placement_both_central_rate", "p1_placement_adjacent_rate", "p1_placement_moderate_separation_rate", "p1_placement_far_separation_rate"],
    ["p1_policy_expected_center_distance", "p1_policy_expected_worker_separation", "p1_policy_center_mass", "p1_policy_inner_ring_mass", "p1_policy_outer_ring_mass"],
    ["p2_placement_mean_center_distance", "p2_winner_mean_center_distance", "p2_loser_mean_center_distance"],
    ["p2_placement_mean_worker_separation", "p2_winner_mean_worker_separation", "p2_loser_mean_worker_separation"],
    ["p2_placement_both_central_rate", "p2_placement_adjacent_rate", "p2_placement_moderate_separation_rate", "p2_placement_far_separation_rate", "p2_placement_adjacent_to_p1_rate"],
    ["p2_policy_expected_center_distance", "p2_policy_expected_worker_separation", "p2_placement_mean_nearest_p1_distance", "placement_mean_minimum_opponent_distance"],
    ["p2_policy_center_mass", "p2_policy_inner_ring_mass", "p2_policy_outer_ring_mass"],
    ["placement_center_owned_by_p1_rate", "placement_center_owned_by_p2_rate", "placement_center_unoccupied_rate", "p2_first_placement_center_available_rate", "p2_policy_center_mass_when_available"],
    ["milestone_current_win_rate", "placement_milestone_current_win_rate", "anchor_current_win_rate"],
    ["training_steps", "actual_replay_reuse", "base_replay_epochs"],
    ["playout_cap_full_search_rate", "playout_cap_average_simulations"],
    ["standard_policy_target_entropy", "placement_policy_target_entropy"],
    ["standard_policy_target_support", "placement_policy_target_support"],
    ["standard_policy_target_one_hot_rate", "placement_policy_target_one_hot_rate"],
    ["tactical_immediate_win_roots", "tactical_single_forced_block_roots", "tactical_forced_block_pruned_roots", "tactical_proven_loss_in_two_roots", "tactical_simulations_skipped"],
]
fig, axes = plt.subplots(16, 2, figsize=(14, 64))
for axis, columns in zip(axes.flat, groups):
    available = [column for column in columns if column in metrics]
    if available: metrics.plot(x="iteration", y=available, ax=axis, marker="o")
for axis in axes.flat[len(groups):]: axis.set_visible(False)
plt.tight_layout()

## 8. Non-gating standard-play evaluation against greedy
Uses 20 fixed, distinct completed openings with both seat assignments. V3 receives a fresh MCTS tree for every game; placement strength is measured separately by milestone telemetry.

In [ ]:
subprocess.run([sys.executable, "pit_santorini.py", "--architecture", "v3", "--checkpoint-folder", CHECKPOINT, "--checkpoint-file", "latest.pth.tar", "--baseline", "greedy", "--opening-source", "unique", "--opening-seed", "20260715", "--games", "40", "--sims", "64", "--json-out", os.path.join(TELEMETRY_DIR, "eval_greedy_standard_40_s64.json")], check=True)